# 02 · Exploración del stream de eventos de uso

**Proyecto:** Cloud Provider Analytics · **Instancia:** primera entrega (diseño y fundación) · **Fuente:** `datalake/landing/usage_events_stream/*.jsonl`

Chequeos de **consistencia**: estructura y evolución de esquema (v1/v2), tipos de dato, dominios, nulos, unicidad de `event_id`, coherencia con los maestros y orden temporal de llegada.
No se analizan distribuciones ni valores de negocio (promedios, totales, percentiles): eso corresponde a Silver/Gold.
Landing **no se modifica**: este notebook solo lee.

**Cómo ejecutar:** igual que el notebook 01 (venv activo, VS Code o `jupyter lab`). Los ≈13 MB de eventos se leen en pocos segundos con pandas; PySpark y Structured Streaming se usan desde la segunda entrega.

## 0 · Setup

Busca `datalake/landing/` subiendo desde la carpeta actual. Si el notebook corre fuera del repo, definí la variable de entorno `LANDING_DIR` con la ruta a los datos.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 60)


def find_landing() -> Path:
    """Devuelve la carpeta datalake/landing (solo lectura: Landing es inmutable)."""
    env = os.environ.get("LANDING_DIR")
    if env and Path(env).is_dir():
        return Path(env)
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        cand = base / "datalake" / "landing"
        if cand.is_dir():
            return cand
    raise FileNotFoundError(
        "No se encontró datalake/landing/. Ejecutá el notebook desde el repo o definí la variable "
        "de entorno LANDING_DIR con la ruta a los datos."
    )


LANDING = find_landing()
n_files = sum(1 for _ in LANDING.rglob("*") if _.is_file())
print(f"Landing encontrado ({n_files} archivos). pandas {pd.__version__}")

Landing encontrado (127 archivos). pandas 3.0.5


## 1 · Carga y volumen

Cada archivo JSONL se lee tal cual; se agregan columnas técnicas (`_file`, `_file_idx`, `_line`), equivalentes al `source_file` previsto para Bronze. También se guarda el conjunto de claves de cada línea para estudiar el esquema.

In [2]:
archivos = sorted((LANDING / "usage_events_stream").glob("*.jsonl"))
filas, claves_por_linea, lineas_invalidas = [], [], 0
for i, f in enumerate(archivos):
    with open(f, encoding="utf-8") as fh:
        for n, linea in enumerate(fh):
            try:
                d = json.loads(linea)
            except json.JSONDecodeError:
                lineas_invalidas += 1
                continue
            claves_por_linea.append(frozenset(d))
            d["_file"], d["_file_idx"], d["_line"] = f.name, i, n
            filas.append(d)

ev = pd.DataFrame(filas)
print(f"archivos             : {len(archivos)}")
print(f"líneas no parseables : {lineas_invalidas}")
print(f"eventos              : {len(ev):,}")
print(f"tamaño total         : {sum(f.stat().st_size for f in archivos) / 1e6:.1f} MB")
print(f"eventos por archivo  : {[int(n) for n in sorted(ev.groupby('_file_idx').size().unique())]}")

archivos             : 120
líneas no parseables : 0
eventos              : 43,200
tamaño total         : 12.9 MB
eventos por archivo  : [360]


## 2 · Estructura y evolución de esquema (v1 → v2)

Se compara el conjunto de claves presentes en cada línea con `schema_version`. La diferencia entre una clave **ausente** y una clave presente con `null` importa: Spark las trata distinto al inferir el esquema.

In [3]:
esquemas = pd.Series(claves_por_linea).map(lambda s: ", ".join(sorted(s))).rename("claves presentes")
tabla = pd.DataFrame({"schema_version": ev.schema_version, "claves presentes": esquemas.values})
resumen = tabla.groupby(["schema_version", "claves presentes"]).size().rename("eventos").reset_index()
for _, r in resumen.iterrows():
    print(f"v{r.schema_version} · {r.eventos:>6} eventos · {r['claves presentes']}")
print()
print("schema_version, valores distintos:", sorted(ev.schema_version.unique().tolist()))

v1 ·  10800 eventos · cost_usd_increment, event_id, metric, org_id, region, resource_id, schema_version, service, timestamp, unit, value
v2 ·   3132 eventos · carbon_kg, cost_usd_increment, event_id, genai_tokens, metric, org_id, region, resource_id, schema_version, service, timestamp, unit, value
v2 ·  29268 eventos · carbon_kg, cost_usd_increment, event_id, metric, org_id, region, resource_id, schema_version, service, timestamp, unit, value

schema_version, valores distintos: [1, 2]


In [4]:
campos = ["event_id", "timestamp", "org_id", "resource_id", "service", "region", "metric", "value", "unit",
          "cost_usd_increment", "carbon_kg", "genai_tokens"]
presentes = pd.DataFrame({c: [sum(1 for s in claves_por_linea if c in s)] for c in campos}, index=["clave presente"]).T
presentes["no nulos"] = ev[campos].count()
presentes["nulos explícitos"] = presentes["clave presente"] - presentes["no nulos"]
presentes["clave ausente"] = len(ev) - presentes["clave presente"]
presentes

,clave presente,no nulos,nulos explícitos,clave ausente
event_id,43200,43200,0,0
timestamp,43200,43200,0,0
org_id,43200,43200,0,0
resource_id,43200,43200,0,0
service,43200,43200,0,0
region,43200,43200,0,0
metric,43200,43200,0,0
value,43200,42323,877,0
unit,43200,41125,2075,0
cost_usd_increment,43200,43200,0,0


In [5]:
v1, v2 = ev[ev.schema_version == 1], ev[ev.schema_version == 2]
print("v1 con carbon_kg / genai_tokens        :", int(v1.carbon_kg.notna().sum()), "/", int(v1.genai_tokens.notna().sum()))
print("v2 sin carbon_kg                       :", int(v2.carbon_kg.isna().sum()), "de", len(v2))
print("v2 eventos genai / con genai_tokens    :", int((v2.service == "genai").sum()), "/", int(v2.genai_tokens.notna().sum()))
print("genai_tokens en servicios no genai     :", int((ev.service.ne("genai") & ev.genai_tokens.notna()).sum()))
print("carbon_kg == 0 (cero explícito)        :", int((ev.carbon_kg == 0).sum()), "| ausente:", int(ev.carbon_kg.isna().sum()), "(distinguir ausencia de cero)")

v1 con carbon_kg / genai_tokens        : 0 / 0
v2 sin carbon_kg                       : 0 de 32400
v2 eventos genai / con genai_tokens    : 3132 / 3132
genai_tokens en servicios no genai     : 0
carbon_kg == 0 (cero explícito)        : 2641 | ausente: 10800 (distinguir ausencia de cero)


## 3 · Tipos de dato

Se mide el tipo **JSON** de cada campo tal como llega (no el que infiere pandas). Lo importante: `value` puede venir como número, texto o `null`, y `timestamp` debe ser ISO 8601 en UTC.

In [6]:
from collections import Counter

# Se cuenta sobre los diccionarios originales: el DataFrame convertiría null en NaN (float) y taparía los tipos reales.
tipos = {c: Counter(type(d[c]).__name__ if c in d else "(ausente)" for d in filas) for c in campos}
pd.DataFrame({"tipos JSON observados": {c: ", ".join(f"{k}: {v}" for k, v in t.most_common()) for c, t in tipos.items()}})

,tipos JSON observados
event_id,str: 43200
timestamp,str: 43200
org_id,str: 43200
resource_id,str: 43200
service,str: 43200
region,str: 43200
metric,str: 43200
value,"float: 41014, str: 1309, NoneType: 877"
unit,"str: 41125, NoneType: 2075"
cost_usd_increment,float: 43200


In [7]:
valor_num = pd.to_numeric(ev.value, errors="coerce")
es_texto = ev.value.map(lambda x: isinstance(x, str))
print("value como texto        :", int(es_texto.sum()))
print("  convertibles a número :", int((es_texto & valor_num.notna()).sum()))
print("  NO convertibles       :", int((es_texto & valor_num.isna()).sum()))
print()
patron_ts = r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$"
ts_ok = ev.timestamp.astype(str).str.match(patron_ts)
ev["ts"] = pd.to_datetime(ev.timestamp, utc=True, errors="coerce")
print("timestamp con formato ISO-8601 UTC (…Z) :", int(ts_ok.sum()), "de", len(ev))
print("timestamp no parseable                  :", int(ev.ts.isna().sum()))
print("timestamp con segundos distintos de :00 :", int((ev.ts.dt.second != 0).sum()))
print()
print("event_id con formato evt_<12 alfanum>   :", int(ev.event_id.astype(str).str.match(r"^evt_[a-z0-9]{12}$").sum()), "de", len(ev))
print("org_id con formato org_<8 alfanum>      :", int(ev.org_id.astype(str).str.match(r"^org_[a-z0-9]{8}$").sum()), "de", len(ev))
print("resource_id con formato res_<8 alfanum> :", int(ev.resource_id.astype(str).str.match(r"^res_[a-z0-9]{8}$").sum()), "de", len(ev))
print("genai_tokens no entero                  :", int((ev.genai_tokens.dropna() % 1 != 0).sum()))

value como texto        : 1309
  convertibles a número : 1309
  NO convertibles       : 0

timestamp con formato ISO-8601 UTC (…Z) : 43200 de 43200
timestamp no parseable                  : 0
timestamp con segundos distintos de :00 : 0

event_id con formato evt_<12 alfanum>   : 43200 de 43200
org_id con formato org_<8 alfanum>      : 43200 de 43200
resource_id con formato res_<8 alfanum> : 43200 de 43200
genai_tokens no entero                  : 0


## 4 · Dominios y relaciones entre campos

Valores permitidos de los campos categóricos y coherencia entre `metric` y `unit`.

In [8]:
for c in ["service", "region", "metric", "unit"]:
    print(f"{c:8}:", ev[c].value_counts(dropna=False).to_dict())

service : {'compute': 12498, 'storage': 7614, 'database': 7419, 'networking': 6921, 'analytics': 4590, 'genai': 4158}
region  : {'ap-northeast': 7773, 'us-east': 7566, 'us-west': 6384, 'sa-east': 6270, 'ap-south': 5943, 'eu-west': 4854, 'eu-central': 4410}
metric  : {'requests': 19512, 'storage_gb_hours': 12996, 'cpu_hours': 10692}
unit    : {'count': 18596, 'gb_hours': 12342, 'hours': 10187, nan: 2075}


In [9]:
print("Cada métrica, ¿con qué unidades aparece? (NaN = unit nulo)")
ev.assign(unit=ev.unit.fillna("NaN")).groupby("metric").unit.agg(lambda s: dict(s.value_counts())).to_frame("unidades")

Cada métrica, ¿con qué unidades aparece? (NaN = unit nulo)


,unidades
metric,
cpu_hours,"{'hours': 10187, 'NaN': 505}"
requests,"{'count': 18596, 'NaN': 916}"
storage_gb_hours,"{'gb_hours': 12342, 'NaN': 654}"


In [10]:
print("Combinaciones servicio × métrica (el dominio de metric no depende del servicio):")
pd.crosstab(ev.service, ev.metric)

Combinaciones servicio × métrica (el dominio de metric no depende del servicio):


metric,cpu_hours,requests,storage_gb_hours
service,,,
analytics,1148,2075,1367
compute,3143,5570,3785
database,1767,3357,2295
genai,1076,1844,1238
networking,1661,3210,2050
storage,1897,3456,2261


## 5 · Reglas de calidad candidatas

Reglas del tipo exigido para la segunda entrega (`event_id` no nulo y único, `cost_usd_increment >= -0.01` con flag de anomalía, `unit` no nulo cuando existe `value`), más chequeos adicionales de consistencia. Se miden acá para dimensionar el quarantine; todavía no se aplican.

In [11]:
neg = ev.cost_usd_increment < 0
reglas = {
    "event_id nulo": ev.event_id.isna(),
    "event_id repetido": ev.event_id.duplicated(keep=False),
    "timestamp no parseable": ev.ts.isna(),
    "campos clave nulos (org_id, resource_id, service, region, metric)": ev[["org_id", "resource_id", "service", "region", "metric"]].isna().any(axis=1),
    "cost_usd_increment nulo": ev.cost_usd_increment.isna(),
    "cost_usd_increment < -0.01  (flag de anomalía)": ev.cost_usd_increment < -0.01,
    "cost_usd_increment en [-0.01, 0)  (tolerado)": neg & (ev.cost_usd_increment >= -0.01),
    "value nulo": ev.value.isna(),
    "value texto no convertible": es_texto & valor_num.isna(),
    "value negativo": valor_num < 0,
    "unit nulo con value presente": ev.unit.isna() & ev.value.notna(),
    "unit nulo sin value": ev.unit.isna() & ev.value.isna(),
    "carbon_kg negativo": ev.carbon_kg < 0,
    "genai_tokens negativo": ev.genai_tokens < 0,
}
tabla_reglas = pd.DataFrame({"registros": {k: int(v.sum()) for k, v in reglas.items()}})
tabla_reglas["% del total"] = (tabla_reglas.registros / len(ev) * 100).round(2)
tabla_reglas

,registros,% del total
event_id nulo,0,0.00
event_id repetido,0,0.00
timestamp no parseable,0,0.00
"campos clave nulos (org_id, resource_id, service, region, metric)",0,0.00
cost_usd_increment nulo,0,0.00
cost_usd_increment < -0.01 (flag de anomalía),211,0.49
"cost_usd_increment en [-0.01, 0) (tolerado)",5,0.01
value nulo,877,2.03
value texto no convertible,0,0.00
value negativo,0,0.00


In [12]:
cols_muestra = ["event_id", "timestamp", "service", "metric", "value", "unit", "cost_usd_increment", "schema_version", "_file"]
print("Muestra: cost_usd_increment < -0.01")
print(ev.loc[reglas["cost_usd_increment < -0.01  (flag de anomalía)"], cols_muestra].head(5).to_string(index=False))
print()
print("Muestra: unit nulo con value presente")
print(ev.loc[reglas["unit nulo con value presente"], cols_muestra].head(5).to_string(index=False))

Muestra: cost_usd_increment < -0.01
        event_id            timestamp  service           metric   value     unit  cost_usd_increment  schema_version                  _file
evt_bbmth9hzpa6e 2025-07-03T03:08:00Z  compute        cpu_hours  3.4886    hours             -0.3656               1 events_part_0000.jsonl
evt_faubjbtabmwl 2025-08-09T23:10:00Z database        cpu_hours  1.1887    hours             -0.0602               2 events_part_0001.jsonl
evt_qniow8ymxwd6 2025-08-25T06:26:00Z  compute        cpu_hours  2.7166    hours             -0.2446               2 events_part_0001.jsonl
evt_whpppwj15cp3 2025-07-13T06:03:00Z  storage        cpu_hours   2.262    hours             -0.0542               1 events_part_0002.jsonl
evt_auh2c9r94utf 2025-07-15T20:18:00Z    genai storage_gb_hours   2.204 gb_hours             -0.2471               1 events_part_0003.jsonl

Muestra: unit nulo con value presente
        event_id            timestamp    service           metric    value unit  cost

## 6 · Unicidad e integridad con los maestros

`event_id` debe ser único. Cada evento debe referir a un recurso y una organización existentes, y `service`/`region`/`org_id` deben coincidir con los del recurso. Además, un evento no debería ser anterior a la creación del recurso ni al alta de la organización.

In [13]:
print("event_id repetidos         :", int(ev.event_id.duplicated().sum()))
cols_dup = [c for c in ev.columns if c not in ("event_id", "_file", "_file_idx", "_line", "ts")]
print("filas idénticas sin event_id  :", int(ev[cols_dup].astype(str).duplicated().sum()))
print()
res = pd.read_csv(LANDING / "resources.csv")
cli = pd.read_csv(LANDING / "customers_orgs.csv")
print("resource_id duplicado en resources.csv:", int(res.resource_id.duplicated().sum()))
j = ev.merge(res[["resource_id", "org_id", "service", "region", "created_at"]], on="resource_id", how="left", suffixes=("", "_recurso"))
sin_recurso = j.org_id_recurso.isna()
print("resource_id inexistente en resources.csv :", int(sin_recurso.sum()))
print("org_id distinto del dueño del recurso    :", int((j.org_id != j.org_id_recurso)[~sin_recurso].sum()))
print("service distinto del recurso             :", int((j.service != j.service_recurso)[~sin_recurso].sum()))
print("region distinta del recurso              :", int((j.region != j.region_recurso)[~sin_recurso].sum()))
print("org_id inexistente en customers_orgs.csv :", int((~ev.org_id.isin(set(cli.org_id))).sum()))
print()
dia = ev.ts.dt.tz_localize(None).dt.normalize()
n_antes_recurso = int((dia < pd.to_datetime(j.created_at)).sum())
print("evento anterior a created_at del recurso      :", n_antes_recurso, f"({n_antes_recurso / len(ev) * 100:.1f} %)")
print("evento anterior a signup_date de la org       :", int((dia < ev.org_id.map(cli.set_index('org_id').signup_date.pipe(pd.to_datetime))).sum()))

event_id repetidos         : 0
filas idénticas sin event_id  : 0

resource_id duplicado en resources.csv: 0
resource_id inexistente en resources.csv : 0
org_id distinto del dueño del recurso    : 0
service distinto del recurso             : 0
region distinta del recurso              : 0
org_id inexistente en customers_orgs.csv : 0

evento anterior a created_at del recurso      : 7371 (17.1 %)
evento anterior a signup_date de la org       : 0


## 7 · Orden temporal de llegada

El stream está dividido en archivos que simulan micro-lotes. Si cada archivo cubriera un tramo de tiempo, el orden de llegada coincidiría con el orden temporal. Se verifica si es así y qué implicaría para un watermark.

In [14]:
span = ev.groupby("_file_idx").ts.agg(["min", "max"])
span["dias"] = (span["max"] - span["min"]).dt.total_seconds() / 86400
print("Cobertura temporal de cada archivo (días entre su evento más viejo y el más nuevo):")
print(span.dias.describe().round(1)[["min", "50%", "max"]].to_string())
print()
print("archivos cuyos eventos están ordenados por timestamp:", sum(g.ts.is_monotonic_increasing for _, g in ev.groupby("_file_idx")), "de", len(archivos))

def tardios(tolerancia):
    max_visto, total = None, 0
    for _, lote in ev.groupby("_file_idx", sort=True):
        if max_visto is not None:
            total += int((lote.ts < max_visto - tolerancia).sum())
        max_visto = lote.ts.max() if max_visto is None else max(max_visto, lote.ts.max())
    return total

t7 = tardios(pd.Timedelta(days=7))
print(f"eventos tardíos con watermark de 7 días (simulación por archivo): {t7} ({t7 / len(ev) * 100:.1f} %)")

Cobertura temporal de cada archivo (días entre su evento más viejo y el más nuevo):
min    59.0
50%    59.8
max    60.0

archivos cuyos eventos están ordenados por timestamp: 0 de 120
eventos tardíos con watermark de 7 días (simulación por archivo): 37831 (87.6 %)


**Lectura.** Cada archivo abarca todo el período, así que el orden de llegada no es temporal y, apenas se procesa el primer lote, el watermark queda cerca del final de los datos. En este dataset el watermark **no sirve para descartar datos**; la idempotencia debe apoyarse en otros mecanismos (upsert por clave natural, recomputo desde Silver). La estrategia final se define en la implementación.

## 8 · Resumen de hallazgos

Tabla generada con los valores calculados arriba. La columna de tratamiento es una **propuesta**.

In [15]:
incoherentes = int((~sin_recurso & ((j.org_id != j.org_id_recurso) | (j.service != j.service_recurso) | (j.region != j.region_recurso))).sum()) + int(sin_recurso.sum())
hallazgos = [
    ("volumen", f"{len(ev):,} eventos en {len(archivos)} archivos", len(ev), "Parquet particionado por event_date"),
    ("esquema", "eventos con schema_version = 1 (sin carbon_kg ni genai_tokens)", int((ev.schema_version == 1).sum()), "Esquema común con campos opcionales; no imputar 0"),
    ("esquema", "eventos con schema_version = 2", int((ev.schema_version == 2).sum()), "carbon_kg siempre informado; genai_tokens solo en servicio genai"),
    ("tipos", "value como texto (todo convertible a número)", int(es_texto.sum()), "Cast con fallback controlado en Bronze/Silver"),
    ("tipos", "timestamp fuera de formato ISO-8601 UTC", int((~ts_ok).sum()), "Sin acción: formato uniforme"),
    ("nulos", "value nulo", int(ev.value.isna().sum()), "Conservar; la validación depende de la métrica"),
    ("calidad", "unit nulo con value presente", int(reglas["unit nulo con value presente"].sum()), "Regla de quarantine/flag (unit no nulo cuando existe value)"),
    ("calidad", "cost_usd_increment < -0.01", int(reglas["cost_usd_increment < -0.01  (flag de anomalía)"].sum()), "Flag de anomalía; decidir si son ajustes válidos"),
    ("calidad", "cost_usd_increment en [-0.01, 0)", int(reglas["cost_usd_increment en [-0.01, 0)  (tolerado)"].sum()), "Tolerado por la regla"),
    ("integridad", "eventos anteriores a la fecha de creación de su recurso", n_antes_recurso, "Definir si se rechazan o se conservan con flag (decisión abierta)"),
    ("orden", "cobertura temporal mediana por archivo (días)", int(round(span.dias.median())), "El orden de llegada no es temporal: usar upsert idempotente"),
    ("orden", "eventos tardíos con watermark de 7 días (simulado)", t7, "El watermark no sirve para descartar datos en este dataset"),
    ("unicidad", "event_id repetidos", int(ev.event_id.duplicated().sum()), "Dedup por event_id igualmente (idempotencia ante reprocesos)"),
    ("integridad", "eventos con recurso inexistente o campos incoherentes con el recurso", incoherentes, "Sin acción: coherentes con resources.csv"),
]
pd.DataFrame(hallazgos, columns=["tema", "hallazgo", "registros", "tratamiento propuesto"])

,tema,hallazgo,registros,tratamiento propuesto
0,volumen,"43,200 eventos en 120 archivos",43200,Parquet particionado por event_date
1,esquema,eventos con schema_version = 1 (sin carbon_kg ni genai_t...,10800,Esquema común con campos opcionales; no imputar 0
2,esquema,eventos con schema_version = 2,32400,carbon_kg siempre informado; genai_tokens solo en servic...
3,tipos,value como texto (todo convertible a número),1309,Cast con fallback controlado en Bronze/Silver
4,tipos,timestamp fuera de formato ISO-8601 UTC,0,Sin acción: formato uniforme
5,nulos,value nulo,877,Conservar; la validación depende de la métrica
6,calidad,unit nulo con value presente,2038,Regla de quarantine/flag (unit no nulo cuando existe value)
7,calidad,cost_usd_increment < -0.01,211,Flag de anomalía; decidir si son ajustes válidos
8,calidad,"cost_usd_increment en [-0.01, 0)",5,Tolerado por la regla
9,integridad,eventos anteriores a la fecha de creación de su recurso,7371,Definir si se rechazan o se conservan con flag (decisión...
